# 描述性统计 
描述性统计部分，主要统计Agent的分布，学习曲线等结果  

**说明**   
统计每个node的meta，对于同一个任务，meta中除train_config字段外，其他字段都是一样的，仅需要第一行的内容就行，表示证券共享的字段（如训练环境，奖励计算配置等）；train_config是每个agent的训练参数，随机采样而来，需要统计均值、方差等统计参数。   


## 导入库

In [7]:
import os
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots

## 超参数

In [8]:
TASK_ID_PREFIX = 'test'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

In [9]:
if SAVE: 
    os.makedirs(SAVE_BASE_DIR, exist_ok=True)
    os.makedirs(SAVE_BASELINE_REG_DIR, exist_ok=True)

## 读取agent的meta  

In [10]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

node_paths

['/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988592_15963',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988574_30803',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996658_s16',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996652_s30978',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996659_s3504',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996658_s15936',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996659_s30978',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_t

读取所有node的meta.json，保存为lf  

需要对lf进行过滤： 

- 1.meta的顶层字段，都是一样的，仅需要第一行的内容就行  

- 2.meta的train_config字段，每一个agent都不同，需要单独展开，进行统计  

In [11]:
# meta.json 统一 schema：避免 concat 时 factors_list/列顺序/缺失列 不一致
META_SCHEMA_OVERRIDES = {
    "task_id": pl.Utf8,
    "start_year": pl.Int64,
    "end_year": pl.Int64,
    "end_month": pl.Int64,
    "N": pl.Int64,
    "stock_list": pl.List(pl.Utf8),
    "earliest_year_month": pl.List(pl.Int64),
    "n": pl.Int64,
    "max_portfolios_num": pl.Int64,
    "env_config": pl.Struct([
        pl.Field("sample_and_shuffle_seed", pl.Int64),
        pl.Field("rl_end_year", pl.Int64),
        pl.Field("save_model_every_n_steps", pl.Int64),
        pl.Field("save_record_every_n_steps", pl.Int64),
        pl.Field("save_performance_and_reward_every_n_steps", pl.Int64),
        pl.Field("box_max", pl.Int64),
        pl.Field("box_min", pl.Int64),
        pl.Field("retrain_times", pl.Int64),
    ]),
    "performance_config": pl.Struct([
        pl.Field("risk_free_rate", pl.Float64),
        pl.Field("rolling_window", pl.Int64),
        pl.Field("std_window", pl.Int64),
        pl.Field("std_floor", pl.Float64),
        pl.Field("classic_utility", pl.Boolean),
    ]),
    "short_limit": pl.Float64,
    "checkpoint": pl.Boolean,
    "factors_list": pl.List(pl.Utf8),
    "train_config": pl.Struct([
        pl.Field("seed", pl.Int64),
        pl.Field("m", pl.Int64),
        pl.Field("mask_len", pl.Int64),
        pl.Field("model_config", pl.Struct([
            pl.Field("cate", pl.Int64),
            pl.Field("config", pl.Struct([])),
            pl.Field("dropout", pl.Float64),
        ])),
        pl.Field("reinforcement_config", pl.Struct([
            pl.Field("cate", pl.Int64),
            pl.Field("opt", pl.Struct([
                pl.Field("lr", pl.Float64),
                pl.Field("clip_grad_norm", pl.Float64),
                pl.Field("weight_decay", pl.Float64),
            ])),
        ])),
        pl.Field("reward_config", pl.Struct([
            pl.Field("reward_weights", pl.Struct([
                pl.Field("rtr", pl.Float64),
                pl.Field("vol", pl.Float64),
                pl.Field("sharpe", pl.Float64),
                pl.Field("max_drawdown", pl.Float64),
                pl.Field("diversification", pl.Float64),
            ])),
            pl.Field("A", pl.Float64),
        ])),
        pl.Field("config_uuid", pl.Utf8),
    ]),
    "port": pl.Int64,
}
META_ALL_COLS = list(META_SCHEMA_OVERRIDES.keys())

In [12]:
# 读取数据：逐文件用统一 schema 读，补全缺失列后 concat，避免 schema 不一致
lf_list = []
for node_path in node_paths:
    df = pl.read_ndjson(os.path.join(node_path, 'meta.json'), schema_overrides=META_SCHEMA_OVERRIDES)
    for c in META_ALL_COLS:
        if c not in df.columns:
            df = df.with_columns(pl.lit(None).cast(META_SCHEMA_OVERRIDES[c]).alias(c))
    df = df.select(META_ALL_COLS)
    lf_list.append(df)
lf_df = pl.concat(lf_list, rechunk=True).lazy()

## 描述性统计 
### node个数统计  
统计一共有多少个node  


In [13]:
node_count = len(lf_list)
print(f'一共有 {node_count} 个node，即 {node_count} 个agent')

一共有 15 个node，即 15 个agent


 ### 顶层字段统计  
meta的顶层字段，是所有证券的共同字段，仅需要第一行的内容就行，但需要处理  
- 将task_id字段改为task_prefix，相同的task，prefix一致，但是uuid不同  
- 展开env_config  
- 展开performance_config  

进行的操作：  
1.添加一个start_month字段，默认为1  
2.去掉earliest_year_month字段     
3.去掉save_model_every_n_steps,save_record_every_n_steps,save_performance_and_reward_every_n_steps字段    
4.按照如下字段重排序，转为长表(字段 | 字段含义 | 值)  

顶层字段包括：  
task_prefix 任务前缀  
start_year 起始年份    
start_month 起始月份    
end_year 结束年份    
end_month 结束月份       
stock_list 股票列表   
N 股票数量      
n 资产组合大小      
max_portfolios_num 最大学习的资产组合数量    
factors_list 因子列表    
short_limit 做空/融资限制   
sample_and_shuffle_seed 抽样/打乱种子  
rl_end_year 强化学习结束年份
box_max 输出上限
box_min 输出下限  
risk_free_rate 无风险利率
rolling_window 滚动窗口
std_window 标准窗口
std_floor 标准差下限  

In [14]:
# 顶层字段顺序及中文含义（用于长表）
META_TOP_ORDER = [
    'task_prefix', 'start_year', 'start_month', 'end_year', 'end_month',
    'stock_list', 'N', 'n', 'max_portfolios_num', 'factors_list', 'short_limit',
    'sample_and_shuffle_seed', 'rl_end_year', 'box_max', 'box_min',
    'risk_free_rate', 'rolling_window', 'std_window', 'std_floor',
]
META_TOP_MEANING = {
    'task_prefix': '任务前缀', 'start_year': '起始年份', 'start_month': '起始月份',
    'end_year': '结束年份', 'end_month': '结束月份', 'stock_list': '股票列表',
    'N': '股票数量', 'n': '资产组合大小', 'max_portfolios_num': '最大学习的资产组合数量',
    'factors_list': '因子列表', 'short_limit': '做空/融资限制',
    'sample_and_shuffle_seed': '抽样/打乱种子', 'rl_end_year': '强化学习结束年份',
    'box_max': '输出上限', 'box_min': '输出下限', 'risk_free_rate': '无风险利率',
    'rolling_window': '滚动窗口', 'std_window': '标准窗口', 'std_floor': '标准差下限',
}

# 1. 取第一行，task_id 改为 task_prefix，展开 env_config / performance_config
meta_lf = (
    lf_df.head(1)
    .with_columns(pl.lit(TASK_ID_PREFIX).alias('task_prefix'))
    .select(pl.all().exclude(['train_config', 'task_id']))
    .with_columns(pl.lit(1).alias('start_month'))
    .with_columns(pl.col('env_config').struct.unnest())
    .drop('env_config')
    .with_columns(pl.col('performance_config').struct.unnest())
    .drop('performance_config')
)
# 2. 去掉指定字段
meta_lf = meta_lf.drop([
    'earliest_year_month',
    'save_model_every_n_steps',
    'save_record_every_n_steps',
    'save_performance_and_reward_every_n_steps',
])
# 3. 按约定顺序选列（仅保留存在的列）
cols = [c for c in META_TOP_ORDER if c in meta_lf.collect_schema().names()]
meta_lf = meta_lf.select(cols)
# 4. 转为长表：字段 | 字段含义 | 值（值统一转为字符串便于展示）
def _to_display_value(v):
    if isinstance(v, (list, dict)):
        return str(v)
    return str(v)

meta_wide = meta_lf.collect()
meta_long = pl.DataFrame({
    '字段': cols,
    '字段含义': [META_TOP_MEANING[c] for c in cols],
    '值': [_to_display_value(meta_wide[c][0]) for c in cols],
})
meta_long

字段,字段含义,值
str,str,str
"""task_prefix""","""任务前缀""","""test"""
"""start_year""","""起始年份""","""2022"""
"""start_month""","""起始月份""","""1"""
"""end_year""","""结束年份""","""2024"""
"""end_month""","""结束月份""","""12"""
…,…,…
"""box_min""","""输出下限""","""-1"""
"""risk_free_rate""","""无风险利率""","""0.0"""
"""rolling_window""","""滚动窗口""","""24"""


In [15]:
if SAVE:
    meta_lf.collect().write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-顶层配置.parquet'))

### 处理train_config字段      
train_config字段，是每个agent的训练参数，随机采样而来，需要进行描述性统计   

mlp模型的train_config字段包括：
- m 回看期数  
- mask_len  掩码长度  
- seed 随机种子   
- model_config 
  - cate 0 模型种类 MLP  
  - dropout 神经元丢失率  
  - config 空，无其他配置  
- reinforcement_config
  - opt 
    - lr 学习率  
    - clip_grad_norm 梯度裁剪  
    - weight_decay 权重衰减  
  - reward_config
    - reward_weights 奖励权重 
      - rtr 收益率  
      - vol 波动率  
      - sharpe 夏普率  
      - max_drawdown 最大回撤  
      - diversification 分散化   
  
分别为model_config, reinforcement_config 和 reward_config 进行统计   

首先，获取train_config，并且展开 

In [16]:
# 获取train_config字段  
train_config = lf_df.select('train_config')
train_config = train_config.with_columns(pl.col('train_config').struct.unnest()).select(pl.all().exclude(['train_config']))
train_config.head().collect() # 

seed,m,mask_len,model_config,reinforcement_config,reward_config,config_uuid
i64,i64,i64,struct[3],struct[2],struct[2],str
195,12,60,"{0,{},0.25}","{0,{0.0005,0.5,0.1}}","{{1.0,0.0,0.0,0.0,0.0},2.438234}","""995816bf-98ab-5511-980f-2c1965…"
53,12,60,"{0,{},0.25}","{0,{0.0005,0.5,0.1}}","{{1.0,0.0,0.0,0.0,0.0},2.48533}","""cc24656d-0275-502e-ab09-e992aa…"
1,12,60,"{0,{},0.25}","{0,{0.0005,0.5,0.1}}","{{1.0,0.0,0.0,0.0,0.0},2.380398}","""4a4f6754-1d65-53c2-9b66-f8e8b9…"
945,12,60,"{0,{},0.25}","{0,{0.0005,0.5,0.1}}","{{1.0,0.0,0.0,0.0,0.0},2.42808}","""06ee61b0-ee0d-5a58-80ce-446b3e…"
514,12,60,"{0,{},0.25}","{0,{0.0005,0.5,0.1}}","{{1.0,0.0,0.0,0.0,0.0},3.647155}","""023ade51-f8a4-5307-9e18-f89e23…"


对于这些变量，定义一个统计函数，统计其指标，并且绘制图像  

In [17]:
def stats(lf:pl.LazyFrame,field:str)->tuple[pl.DataFrame, plotly.graph_objects.Figure]:
    """
    对于df种的field字段，统计其均值方差等指标，绘制柱状图
    
    输入： 
    - lf: pl.LazyFrame 
    - field: str 需要统计的field字段  

    输出：
    - df: pl.DataFrame 统计后的数据(mean,std,median,modes,min,max)
    - fig: plotly.graph_objects.Figure 柱状图  
    """
    if field not in lf.collect_schema().names():
        warnings.warn(f'{field} not in lf')
        return 
    
    # 获取field字段 
    field_col = lf.select(field) 

    # 统计  
    field_stats = field_col.select(
        pl.col(field).mean().alias('mean'),
        pl.col(field).std().alias('std'),
        pl.col(field).median().alias('median'),
        pl.col(field).mode().alias('modes'),
        pl.col(field).min().alias('min'),
        pl.col(field).max().alias('max')
    )

    # 转为长表  
    # 假设 field_stats 的列就是 mean, std, median, modes, min, max
    long_field_stats = field_stats.unpivot(
        index=[],  # 不保留任何列，全部参与拉长
        on=["mean", "std", "median", "modes", "min", "max"],
        variable_name="统计量",
        value_name=f"{field} 统计值",
    )

    # 绘制柱状图  
    fig = px.histogram(
        lf.collect().to_pandas(),
        x=field,
        histnorm='probability density',
        title=f'{field} distribution (density)',
    )
    return long_field_stats.collect(), fig

def stats_batch(lf:pl.LazyFrame,fields:list[str])->tuple[pl.DataFrame,plotly.graph_objects.Figure]:
    """
    批量统计多个字段，返回连接后的统计结果和多子图的柱状图  
    """
    field_stats_list = []
    fig_list = []
    for field in fields:
        field_stats, fig = stats(lf, field)
        field_stats_list.append(field_stats)
        fig_list.append(fig)
    
    # 判断空
    if len(field_stats_list) == 0 or len(fig_list) == 0:
        warnings.warn('统计结果为空列表')
        return None, None

    # 合并统计结果
    combined_field_stats = pl.concat(field_stats_list, rechunk=True, how='align')
    
    # 合并图表：n==1 不建子图，直接返回单图；子图加总标题，单图标题用中文
    n = len(fig_list)
    if n == 1:
        fig_combined = fig_list[0]
        fig_combined.update_layout(title_text=f"{fields[0]} 密度分布")
    else:
        n_cols = 2
        n_rows = (n + n_cols - 1) // n_cols
        fig_combined = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=fields, vertical_spacing=0.12, horizontal_spacing=0.08)
        for i, fig in enumerate(fig_list):
            row, col = i // n_cols + 1, i % n_cols + 1
            for trace in fig.data:
                fig_combined.add_trace(trace, row=row, col=col)
        fig_combined.update_layout(title_text="各字段密度分布")

    return combined_field_stats, fig_combined

#### 数据能力统计
数据能力为顶层字段的m和mask_len，需要统计其均值方差等指标，绘制密度图  

In [18]:
# 顶层字段  
top_fields = ['m','mask_len']

# 统计
stats_df, fig = stats_batch(train_config, top_fields)

# 绘制
print('-------------- train_config 顶层字段统计 --------------')
print(stats_df)
fig.show()

-------------- train_config 顶层字段统计 --------------
shape: (6, 3)
┌────────┬──────────┬─────────────────┐
│ 统计量 ┆ m 统计值 ┆ mask_len 统计值 │
│ ---    ┆ ---      ┆ ---             │
│ str    ┆ f64      ┆ f64             │
╞════════╪══════════╪═════════════════╡
│ max    ┆ 12.0     ┆ 60.0            │
│ mean   ┆ 12.0     ┆ 60.0            │
│ median ┆ 12.0     ┆ 60.0            │
│ min    ┆ 12.0     ┆ 60.0            │
│ modes  ┆ 12.0     ┆ 60.0            │
│ std    ┆ 0.0      ┆ 0.0             │
└────────┴──────────┴─────────────────┘


In [19]:
if SAVE:
    stats_df.write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-数据能力.parquet'))
    fig.write_image(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-数据能力.png'))


#### 神经网络构建配置统计    
先提取model_config字段，并且展开   
有cate, dropout, config字段，在主回归中，默认cate为0，即mlp模型，config为空，不统计，仅统计dropout字段  

In [20]:
model_config = train_config.select('model_config')
model_config = model_config.with_columns(pl.col('model_config').struct.unnest()).select(pl.all().exclude(['model_config']))
# model_config = model_config.filter(pl.col('cate') == 0) # 测试阶段不使用  
model_config = model_config.select('dropout')

In [21]:
# 统计dropout字段  
fields = ['dropout']
stats_df, fig = stats_batch(model_config, fields)
print('-------------- model_config dropout字段统计 --------------')
print(stats_df)
fig.show()



-------------- model_config dropout字段统计 --------------
shape: (6, 2)
┌────────┬────────────────┐
│ 统计量 ┆ dropout 统计值 │
│ ---    ┆ ---            │
│ str    ┆ f64            │
╞════════╪════════════════╡
│ mean   ┆ 0.25           │
│ std    ┆ 0.0            │
│ median ┆ 0.25           │
│ modes  ┆ 0.25           │
│ min    ┆ 0.25           │
│ max    ┆ 0.25           │
└────────┴────────────────┘


In [22]:
if SAVE:
    stats_df.write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-神经网络构建配置.parquet'))
    fig.write_image(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-神经网络构建配置.png'))


#### 强化学习配置统计  
基准回归使用的是PPO，cate默认为0  
提取reinforcement_config字段；其中仅有opt字段，展开，统计各个字段的分布，并且绘制密度图    

In [23]:
# 提取opt  
reinforcement_config = train_config.select('reinforcement_config')
reinforcement_config = reinforcement_config.with_columns(pl.col('reinforcement_config').struct.unnest()).select(pl.all().exclude(['reinforcement_config']))
reinforcement_config.head().collect()
opt = reinforcement_config.select('opt')
opt = opt.with_columns(pl.col('opt').struct.unnest()).select(pl.all().exclude(['opt']))


In [24]:
# 统计
fields = ['lr','clip_grad_norm','weight_decay']
stats_df, fig = stats_batch(opt, fields)
print('-------------- reinforcement_config opt字段统计 --------------')
print(stats_df)
fig.show()



-------------- reinforcement_config opt字段统计 --------------
shape: (6, 4)
┌────────┬────────────┬───────────────────────┬─────────────────────┐
│ 统计量 ┆ lr 统计值  ┆ clip_grad_norm 统计值 ┆ weight_decay 统计值 │
│ ---    ┆ ---        ┆ ---                   ┆ ---                 │
│ str    ┆ f64        ┆ f64                   ┆ f64                 │
╞════════╪════════════╪═══════════════════════╪═════════════════════╡
│ max    ┆ 0.0005     ┆ 0.5                   ┆ 0.1                 │
│ mean   ┆ 0.0005     ┆ 0.5                   ┆ 0.1                 │
│ median ┆ 0.0005     ┆ 0.5                   ┆ 0.1                 │
│ min    ┆ 0.0005     ┆ 0.5                   ┆ 0.1                 │
│ modes  ┆ 0.0005     ┆ 0.5                   ┆ 0.1                 │
│ std    ┆ 1.1223e-19 ┆ 0.0                   ┆ 2.8730e-17          │
└────────┴────────────┴───────────────────────┴─────────────────────┘


In [25]:
if SAVE:
    opt.collect().write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-强化学习配置.parquet'))
    fig.write_image(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-强化学习配置.png'))


#### 奖励函数配置统计  
提取reward_config字段，其中仅有reward_weights字段，展开，统计各个字段的分布，并且绘制密度图    




In [26]:
reward_config = train_config.select('reward_config')
reward_config = reward_config.with_columns(pl.col('reward_config').struct.unnest()).select(pl.all().exclude(['reward_config']))
reward_weights = reward_config.select('reward_weights')
reward_weights = reward_weights.with_columns(pl.col('reward_weights').struct.unnest()).select(pl.all().exclude(['reward_weights']))

In [27]:
# 统计
fields = ['rtr','vol','sharpe','max_drawdown','diversification']
stats_df, fig = stats_batch(reward_weights, fields)
print('-------------- reward_weights 字段统计 --------------')
print(stats_df)
fig.show()

-------------- reward_weights 字段统计 --------------
shape: (6, 6)
┌────────┬────────────┬────────────┬───────────────┬─────────────────────┬────────────────────────┐
│ 统计量 ┆ rtr 统计值 ┆ vol 统计值 ┆ sharpe 统计值 ┆ max_drawdown 统计值 ┆ diversification 统计值 │
│ ---    ┆ ---        ┆ ---        ┆ ---           ┆ ---                 ┆ ---                    │
│ str    ┆ f64        ┆ f64        ┆ f64           ┆ f64                 ┆ f64                    │
╞════════╪════════════╪════════════╪═══════════════╪═════════════════════╪════════════════════════╡
│ max    ┆ 1.0        ┆ 0.0        ┆ 0.0           ┆ 0.0                 ┆ 0.0                    │
│ mean   ┆ 1.0        ┆ 0.0        ┆ 0.0           ┆ 0.0                 ┆ 0.0                    │
│ median ┆ 1.0        ┆ 0.0        ┆ 0.0           ┆ 0.0                 ┆ 0.0                    │
│ min    ┆ 1.0        ┆ 0.0        ┆ 0.0           ┆ 0.0                 ┆ 0.0                    │
│ modes  ┆ 1.0        ┆ 0.0        ┆ 0.0           ┆ 0

In [28]:
if SAVE:
    reward_weights.collect().write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-奖励函数配置.parquet'))
    fig.write_image(os.path.join(SAVE_BASELINE_REG_DIR, '基准回归-奖励函数配置.png'))
